# Imports

In [5]:
import os
import glob
import sqlite3
import pandas as pd

# Inputs

In [6]:
DB_PATH   = "../data/sqlite/data.db"
DATA_ROOT = "../data/raw"
DATASET   = "CIC_IIoT_dataset_2025"

# Load dataset CSVs into database

In [7]:
def load_dataset_to_db(data_root: str, dataset: str, db_path: str, chunksize: int = 50_000) -> None:
    dataset_path = os.path.join(data_root, dataset)
    table_name = dataset.replace("-", "_").replace(" ", "_")

    csv_paths = sorted(
        glob.glob(os.path.join(dataset_path, "**", "merged_*.csv"), recursive=True)
    )

    if not csv_paths:
        print(f"[{dataset}] No merged CSVs found — skipping.")
        return

    os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
    conn = sqlite3.connect(db_path)

    existing = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?", (table_name,)
    ).fetchone()

    if existing:
        print(f"[{dataset}] Table '{table_name}' already exists — skipping.")
        conn.close()
        return

    print(f"[{dataset}] {len(csv_paths)} CSV file(s) → table '{table_name}'")

    for csv_path in csv_paths:
        print(f"  {os.path.basename(csv_path)} ...", end=" ", flush=True)
        for chunk in pd.read_csv(csv_path, chunksize=chunksize):
            chunk.to_sql(table_name, conn, if_exists="append", index=False)
        conn.commit()
        print("✓")

    conn.execute(f'CREATE INDEX IF NOT EXISTS "idx_{table_name}_label" ON "{table_name}" (label)')
    conn.commit()

    total = conn.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    print(f"\n✓ Table '{table_name}': {total:,} rows")
    conn.close()

In [8]:
datasets = [
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
]

In [9]:
for dataset in sorted(datasets):
    load_dataset_to_db(DATA_ROOT, dataset, DB_PATH)

[CIC_IIoT_dataset_2025] 8 CSV file(s) → table 'CIC_IIoT_dataset_2025'
  merged_benign.csv ... ✓
  merged_bruteforce.csv ... ✓
  merged_ddos.csv ... ✓
  merged_dos.csv ... ✓
  merged_output.csv ... ✓
  merged_malware.csv ... ✓
  merged_mitm.csv ... ✓
  merged_web.csv ... ✓

✓ Table 'CIC_IIoT_dataset_2025': 345,667,601 rows
